# 01 — Data exploration

Inspect the fleet sensor series, missing values, and degradation trends.


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_preprocessing import SENSOR_COLUMNS, ensure_raw_data, load_raw

sns.set_theme(style="whitegrid")
path = ensure_raw_data()
df = load_raw(path)
df.head()

In [ ]:
print(df.shape)
print(df.dtypes)
df.describe()

In [ ]:
print("Units:", df["unit_id"].nunique())
print("Cycles per unit:")
print(df.groupby("unit_id")["cycle"].max().describe())
df.isna().sum()

In [ ]:
fig, axes = plt.subplots(len(SENSOR_COLUMNS), 1, figsize=(10, 12), sharex=True)
sample_units = sorted(df["unit_id"].unique())[:4]
for ax, col in zip(axes, SENSOR_COLUMNS):
    for uid in sample_units:
        sub = df[df["unit_id"] == uid]
        ax.plot(sub["cycle"], sub[col], alpha=0.8, label=f"unit {uid}")
    ax.set_ylabel(col)
axes[0].legend(loc="upper left", ncol=4)
axes[-1].set_xlabel("cycle")
fig.suptitle("Sensor trajectories (sample units)")
plt.tight_layout()
plt.show()

In [ ]:
if "RUL" in df.columns:
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(df.groupby("unit_id")["RUL"].max(), bins=15, color="steelblue")
    ax.set_xlabel("Life length (cycles)")
    ax.set_title("Asset lifetime distribution")
    plt.show()

In [ ]:
corr = df[SENSOR_COLUMNS + (["RUL"] if "RUL" in df.columns else [])].corr()
plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0)
plt.title("Feature correlations")
plt.show()